[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [9]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
      N, D  = X.shape
      # 造一列全 1 + 按列拼接
      ones = torch.ones((N, 1), dtype=X.dtype) # N, D
      X_aug = torch.cat([X, ones], dim=1) # N, D+1

      # y.unsqueeze(1) : N--> N,1 因为 lstsq 更习惯右边是二维
      # 解出来是 D+1,1 我们要 D+1 所以 squeeze
      # linalg: linear algebra; lstsq: least squares 最小二乘
      solution = torch.linalg.lstsq(X_aug, y.unsqueeze(1)).solution.squeeze(1)


      w = solution[:D]
      b = solution[D]
      return w, b

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""
        N, D = X.shape
        # w is (D, )
        #形状为 (D,) 的张量（Tensor）既不是“严格意义”上的横向量，也不是纵向量。它被称为 一维张量（1D Tensor），在 PyTorch 中它是没有方向感的。
        # 在乘法里可以左右横跳来适配
        w = torch.zeros(D, dtype=X.dtype, device=X.device)
        b = torch.tensor(0.0, dtype=X.dtype, device=X.device)
        for _ in range(steps):
          # N,D @ D, --> (N, )
          pred = X @ w + b
          error = pred - y
          # 题目中告诉的公式
          # loss是 MSE： 1/N * sum(Y_hat - Y)
          grad_w = (2.0 / N) * (X.T @ error)
          grad_b = (2.0 / N) * error.sum()

          w = w - lr * grad_w
          b = b - lr * grad_b
        return w, b

    # pytorch 自己写好的线性层、可以自动求导
    # 我只需要负责 forward - loss - backward（传送梯度） - step（更新参数）
    # 最常见
    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        N, D = X.shape
        # input 维度是 D， 输出是 1
        # y = X @ w + b
        model = nn.Linear(D, 1, bias=True).to(dtype=X.dtype, device=X.device)

        criterion = nn.MSELoss()
        # SGD = stochastic gradient descent
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

        y_target = y.unsqueeze(1)

        for _ in range(steps):
          # 清空上一个梯度 pytorch 会默认累计梯度
          optimizer.zero_grad()
          pred = model(X)
          loss = criterion(pred, y_target)

          loss.backward()
          optimizer.step()

        # shape 是 (1, D)，因为它是“1 个输出神经元，对应 D 个输入权重
        # detach 是把 tensor 从计算图里拿出来  不再追踪梯度
        w = model.weight.detach().squeeze(0)
        # shape 是 (1,)，我们想要标量，所以也 squeeze(0)。
        b = model.bias.detach().squeeze(0)
        return w, b


In [10]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [11]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
  ✅ [1/6] Closed-form returns correct shapes (2.0ms)
  ✅ [2/6] Closed-form finds correct weights (1.5ms)
  ✅ [3/6] Gradient descent converges (72.5ms)
  ✅ [4/6] nn.Linear approach works (495.0ms)
  ✅ [5/6] All three methods agree (856.2ms)
  ✅ [6/6] Closed-form uses no autograd (0.7ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (1427.9ms total)
  Progress saved. Run status() to see your dashboard.

